# Column pooling of the per-dataset fields

**Question.** Issue [#16](https://github.com/adkipnis/tabdgp/issues/16) draws ten fields of the hyperprior per column or per node instead of once per dataset:
the seven marginal parameters, `level_alpha`, `noise_scale` and `init_scale`.
Four designs pool them differently across a dataset's columns and nodes (units):

| pooling | design | one unit's value |
|---|---|---|
| `today` | T, full pooling | the dataset's draw c ~ D, shared by every unit |
| `independent` | A, no pooling | its own draw from D |
| `alike` | B-alike, a Gaussian copula | z = ρ z_c + √(1 − ρ²) ε with z_c = Φ⁻¹(F_D(c)), mapped back by F_D⁻¹(Φ(z)) |
| `share` | B-share, a mixture | c with chance ρ, else its own draw from D |

D is the field's distribution in the hyperprior, c its draw for the dataset,
and ρ ~ Uniform(0, 1) one alikeness per dataset, shared by the ten fields.
Every design leaves each unit's law exactly D; ρ = 1 is T and ρ = 0 is A.
A flattens: the tables differ less from each other and their columns more among themselves.
Does that show at the table level, where a classifier two-sample test (C2ST) on per-table summaries looks,
or do the graph and the family shares wash it out? And what does each design cost in `src/`?

**Setup.** `core.py` beside this notebook reruns `tabdgp.draw`'s pipeline from its public steps
(`resolve`, `plan`, `connect`, `apply`) and redoes only what reads the ten fields, with a `pooling` argument.
Under `today` it reproduces `tabdgp.draw` bit for bit.
The per-unit values come from a generator of their own, so the column plan, the graph and the node noise
are the same draws under every pooling: a difference comes from the pooling alone.
A fifth pooling, `median`, sets every unit to its field's median, the floor the graph and the family shares leave.
2,000 datasets per pooling, `n_obs = 1000` (an override, so that skewness and kurtosis are stable),
every other field at the working tree's defaults of 2026-10-12 (`latent_nodes` 0 to 9, `design_columns` 1 to 19).
It runs against `f563704`, the last commit before #16 changed `apply` and the draw order.

In [1]:
import math
import platform
import sys
import time

import numpy as np
from core import COLUMN_FIELDS, NODE_FIELDS, SUMMARIES, cdf, drawPooled, measure, spread

import tabdgp
from tabdgp import Hyperprior

print(platform.platform(), 'Python', sys.version.split()[0], 'numpy', np.__version__)

PRIOR = Hyperprior(n_obs=1000)
SEED = 0
DATASETS = 2000
POOLINGS = ['today', 'independent', 'alike', 'share', 'median']

# today must be tabdgp.draw itself, or every comparison below is against something else
identical = True
for index in range(50):
    ours = drawPooled(SEED, index, 'today', PRIOR)['columns']
    theirs = tabdgp.draw(SEED, index, PRIOR).columns
    identical &= all(np.array_equal(ours[name], theirs[name]) for name in theirs)
print('today reproduces tabdgp.draw bit for bit on 50 datasets:', identical)

macOS-26.7-arm64-arm-64bit-Mach-O Python 3.14.7 numpy 2.5.4
today reproduces tabdgp.draw bit for bit on 50 datasets: True


In [2]:
start = time.perf_counter()
runs = {}
for pooling in POOLINGS:
    runs[pooling] = [measure(SEED, index, pooling, PRIOR) for index in range(DATASETS)]
print(f'{len(POOLINGS)} poolings × {DATASETS} datasets in {time.perf_counter() - start:.0f} s')

5 poolings × 2000 datasets in 21 s


## Parameter level

Per dataset, the mean and the sd of log(parameter) over the units that read it:
`noise_scale` over the non-root nodes, `lognormal_sigma` over the lognormal and zero-inflated lognormal columns.
Across datasets, *between* is the sd of the per-dataset means (how much datasets differ)
and *within* the mean of the per-dataset sds (how much a dataset's units differ), over the datasets with two readers or more.

In [3]:
def logSpread(field: str, pooling: str) -> tuple[float, float]:
    means = []
    sds = []
    for run in runs[pooling]:
        logs = np.log(run['parameters'][field])  # (readers,)
        if len(logs) >= 1:
            means.append(logs.mean())
        if len(logs) >= 2:
            sds.append(logs.std(ddof=1))
    return float(np.std(means, ddof=1)), float(np.mean(sds))


for field in ['noise_scale', 'lognormal_sigma']:
    low, high = getattr(PRIOR, field).support()
    readers = np.array([len(run['parameters'][field]) for run in runs['today']])  # (D,)
    print(
        f'{field}: {readers.mean():.2f} readers per dataset, {(readers >= 1).sum()} datasets '
        f'with one or more, {(readers >= 2).sum()} with two or more; '
        f'sd of log({field}) under D {math.log(high / low) / math.sqrt(12):.3f}'
    )
    print(f'  {"pooling":<12} {"between":>8} {"within":>8}')
    for pooling in POOLINGS:
        between, within = logSpread(field, pooling)
        print(f'  {pooling:<12} {between:>8.3f} {within:>8.3f}')

noise_scale: 9.06 readers per dataset, 1960 datasets with one or more, 1889 with two or more; sd of log(noise_scale) under D 1.329
  pooling       between   within
  today           1.336    0.000
  independent     0.582    1.280
  alike           0.885    1.010
  share           0.897    0.937
  median          0.000    0.000
lognormal_sigma: 1.05 readers per dataset, 890 datasets with one or more, 487 with two or more; sd of log(lognormal_sigma) under D 0.517
  pooling       between   within
  today           0.513    0.000
  independent     0.424    0.478
  alike           0.454    0.350
  share           0.446    0.308
  median          0.000    0.000


### Each unit's law is D

The Kolmogorov–Smirnov (KS) distance between D and the drawn values, taken through D's cdf.
*pooled* takes every reader of every dataset; its values within a dataset are dependent, so it has no bar.
*first* takes the first reader of each dataset, so its values are independent and the 95 % bar is 1.36 / √n.

In [4]:
def ksDistance(u: np.ndarray) -> float:
    # the KS distance of a sample of F_D(x) from Uniform(0, 1); ties count once per value
    u = np.sort(u)  # (n,)
    rank = np.arange(1, len(u) + 1)  # (n,)
    return float(max((rank / len(u) - u).max(), (u - (rank - 1) / len(u)).max()))


DRAWN = POOLINGS[:4]  # median is a reference, not a law
print(
    f'{"field":<16} {"n":>5} {"bar":>6}   '
    + ''.join(f'{p:>12}' for p in DRAWN)
    + '   (pooled | first)'
)
for field in COLUMN_FIELDS + NODE_FIELDS:
    prior = getattr(PRIOR, field)
    cells = []
    for pooling in DRAWN:
        pooled = np.concatenate([run['parameters'][field] for run in runs[pooling]])
        first = np.array(
            [run['parameters'][field][0] for run in runs[pooling] if len(run['parameters'][field])]
        )
        cells.append(f'{ksDistance(cdf(prior, pooled)):.3f}|{ksDistance(cdf(prior, first)):.3f}')
    bar = 1.36 / math.sqrt(len(first))
    print(f'{field:<16} {len(first):>5} {bar:>6.3f}   ' + ''.join(f'{cell:>12}' for cell in cells))

field                n    bar          today independent       alike       share   (pooled | first)
student_df         568  0.057    0.029|0.020 0.026|0.033 0.033|0.032 0.038|0.043
lognormal_sigma    890  0.046    0.022|0.032 0.011|0.029 0.013|0.028 0.027|0.022
zero_share         567  0.057    0.062|0.057 0.014|0.032 0.035|0.034 0.058|0.064
beta_a             572  0.057    0.034|0.025 0.029|0.028 0.023|0.034 0.028|0.032
beta_b             572  0.057    0.075|0.044 0.028|0.032 0.043|0.038 0.060|0.041
mixture_gap        591  0.056    0.043|0.032 0.030|0.034 0.017|0.028 0.031|0.035
mixture_weight     591  0.056    0.044|0.027 0.018|0.037 0.049|0.049 0.035|0.024
level_alpha       1801  0.032    0.019|0.013 0.009|0.017 0.009|0.019 0.013|0.012
noise_scale       1960  0.031    0.030|0.021 0.005|0.015 0.017|0.018 0.020|0.023
init_scale        1694  0.033    0.035|0.024 0.011|0.016 0.010|0.020 0.015|0.016


The same check straight from `spread`, the step that draws the per-unit values:
100,000 datasets of 8 units each, the first unit's KS distance against a bar of 1.36 / √100,000 = 0.0043.

In [5]:
check_rng = np.random.default_rng(2026)
print(f'{"field":<16}' + ''.join(f'{p:>12}' for p in DRAWN[1:]))
for field in ['noise_scale', 'zero_share']:  # a LogUniform and a Uniform
    prior = getattr(PRIOR, field)
    distances = []
    for pooling in DRAWN[1:]:
        first = np.empty(100_000)  # (100000,)
        for k in range(len(first)):
            centre = prior.draw(check_rng)
            first[k] = spread(prior, centre, check_rng.random(), 8, pooling, check_rng)[0]
        distances.append(ksDistance(cdf(prior, first)))
    print(f'{field:<16}' + ''.join(f'{distance:>12.4f}' for distance in distances))

field            independent       alike       share


noise_scale           0.0016      0.0028      0.0019


zero_share            0.0014      0.0022      0.0029


## Table level

Per table, over its observed columns:
the mean and sd of the absolute Spearman correlation over all pairs of columns
(set by the graph and the noise scales, since a marginal is rank-matched to its node; coded columns tie);
the mean and sd over continuous columns of the sample skewness and of the excess kurtosis;
the mean and sd over coded columns of the largest level's share.
A summary a table has too few columns for is missing and left out; the last column counts the tables that have it.

In [6]:
TABLES = {}  # pooling -> (D, 8), one row of SUMMARIES per table
for pooling in POOLINGS:
    TABLES[pooling] = np.array(
        [[run['summaries'][name] for name in SUMMARIES] for run in runs[pooling]]
    )

print('between-table sd of each summary, and its ratio to today')
print(f'{"summary":<20}' + ''.join(f'{p:>18}' for p in POOLINGS) + f'{"tables":>8}')
for j, name in enumerate(SUMMARIES):
    today = np.nanstd(TABLES['today'][:, j], ddof=1)
    cells = []
    for pooling in POOLINGS:
        sd = np.nanstd(TABLES[pooling][:, j], ddof=1)
        cells.append(f'{sd:.3g} ({sd / today:.2f})')
    tables = int(np.isfinite(TABLES['today'][:, j]).sum())
    print(f'{name:<20}' + ''.join(f'{cell:>18}' for cell in cells) + f'{tables:>8}')

between-table sd of each summary, and its ratio to today
summary                          today       independent             alike             share            median  tables
spearman mean             0.197 (1.00)      0.174 (0.88)       0.18 (0.91)      0.179 (0.91)        0.2 (1.01)    1832
spearman sd               0.091 (1.00)     0.0844 (0.93)     0.0835 (0.92)     0.0861 (0.95)     0.0875 (0.96)    1644
skew mean                  1.75 (1.00)       1.42 (0.81)       1.49 (0.85)       1.52 (0.87)       0.65 (0.37)    1501
skew sd                    1.76 (1.00)       1.82 (1.03)       1.71 (0.97)       1.62 (0.92)      0.514 (0.29)    1150
kurtosis mean              32.1 (1.00)       23.6 (0.73)       23.6 (0.73)       24.4 (0.76)       3.68 (0.11)    1501
kurtosis sd                  42 (1.00)       44.9 (1.07)       38.8 (0.92)       34.3 (0.82)       4.84 (0.12)    1150
largest share mean        0.146 (1.00)      0.139 (0.95)      0.144 (0.99)      0.141 (0.96)      0.136 (0.93)

In [7]:
print('5th, 50th and 95th percentile of each summary across tables')
print(f'{"summary":<20}' + ''.join(f'{p:>21}' for p in POOLINGS))
for j, name in enumerate(SUMMARIES):
    cells = []
    for pooling in POOLINGS:
        low, middle, high = np.nanpercentile(TABLES[pooling][:, j], [5, 50, 95])
        cells.append(f'{low:.3g} {middle:.3g} {high:.3g}')
    print(f'{name:<20}' + ''.join(f'{cell:>21}' for cell in cells))

5th, 50th and 95th percentile of each summary across tables
summary                             today          independent                alike                share               median
spearman mean          0.0314 0.272 0.678    0.0305 0.27 0.603   0.0299 0.265 0.616   0.0336 0.262 0.614   0.0312 0.322 0.695
spearman sd            0.0455 0.215 0.355    0.046 0.217 0.348   0.0528 0.213 0.346   0.0492 0.216 0.349   0.0476 0.237 0.365
skew mean               -0.273 0.372 3.96     -0.269 0.47 3.57    -0.269 0.425 3.75    -0.285 0.446 3.76   -0.0494 0.445 1.89
skew sd                 0.0436 0.694 4.72     0.0673 1.03 5.12    0.0557 0.939 4.92    0.0532 0.937 4.91    0.0358 0.815 1.55
kurtosis mean            -1.14 0.958 50.4      -1.14 1.66 44.1      -1.12 1.35 46.5      -1.14 1.46 46.4      -1.07 1.19 7.97
kurtosis sd               0.132 2.32 78.8      0.301 5.66 95.8      0.236 4.36 84.7      0.194 4.41 85.8       0.16 2.99 10.4
largest share mean      0.355 0.631 0.846    0.347 0.639 0

In [8]:
print("mean within-table sd: each 'sd' summary averaged over tables")
print(f'{"summary":<20}' + ''.join(f'{p:>12}' for p in POOLINGS))
for j, name in enumerate(SUMMARIES):
    if name.endswith('sd'):
        means = [np.nanmean(TABLES[pooling][:, j]) for pooling in POOLINGS]
        print(f'{name:<20}' + ''.join(f'{mean:>12.3g}' for mean in means))

mean within-table sd: each 'sd' summary averaged over tables
summary                    today independent       alike       share      median
spearman sd                0.212       0.213       0.212       0.214       0.231
skew sd                     1.33        1.63        1.54        1.51       0.736
kurtosis sd                 16.2          21        19.2        18.3        3.83
largest share sd           0.145       0.161       0.151       0.154       0.149


## Visible to a classifier?

Two views of how far each pooling moves the distribution of table summaries away from today's.

- **Per summary**, the two-sample KS distance to today's, over the same 2,000 structures.
  For scale, today's first 1,000 tables against its last 1,000, whose structures are independent.
- **All summaries at once**, a k-nearest-neighbour (k-NN) C2ST: 1,000 tables of one pooling against 1,000 of another,
  from disjoint halves of the indices so that no structure meets its twin.
  The features are each summary's ranks over both samples, a missing summary at −1;
  the score is the leave-one-out accuracy with k = 25, averaged over 20 random halvings, its range in brackets.
  0.5 is indistinguishable.

In [9]:
def ksTwoSample(first: np.ndarray, second: np.ndarray) -> float:
    first = np.sort(first[np.isfinite(first)])
    second = np.sort(second[np.isfinite(second)])
    grid = np.concatenate([first, second])
    gap = np.searchsorted(first, grid, side='right') / len(first) - np.searchsorted(
        second, grid, side='right'
    ) / len(second)
    return float(np.abs(gap).max())


half = DATASETS // 2
print('KS distance of each summary to today')
print(f'{"summary":<20}' + ''.join(f'{p:>12}' for p in POOLINGS[1:]) + f'{"halves":>12}')
for j, name in enumerate(SUMMARIES):
    today = TABLES['today'][:, j]
    distances = [ksTwoSample(TABLES[pooling][:, j], today) for pooling in POOLINGS[1:]]
    null = ksTwoSample(today[:half], today[half:])
    print(
        f'{name:<20}' + ''.join(f'{distance:>12.3f}' for distance in distances) + f'{null:>12.3f}'
    )

KS distance of each summary to today
summary              independent       alike       share      median      halves
spearman mean              0.052       0.046       0.045       0.108       0.039
spearman sd                0.043       0.041       0.036       0.134       0.060
skew mean                  0.061       0.040       0.047       0.100       0.040
skew sd                    0.141       0.123       0.111       0.225       0.045
kurtosis mean              0.093       0.064       0.065       0.179       0.039
kurtosis sd                0.150       0.110       0.112       0.235       0.058
largest share mean         0.044       0.038       0.024       0.052       0.055
largest share sd           0.125       0.074       0.087       0.045       0.047


In [10]:
def knnAccuracy(first: np.ndarray, second: np.ndarray, k: int = 25) -> float:
    pooled = np.concatenate([first, second])  # (2m, 8)
    labels = np.repeat([0, 1], [len(first), len(second)])  # (2m,)
    features = np.full(pooled.shape, -1.0)  # (2m, 8)
    for j in range(pooled.shape[1]):
        finite = np.isfinite(pooled[:, j])
        features[finite, j] = np.argsort(np.argsort(pooled[finite, j])) / finite.sum()
    squared = (features**2).sum(axis=1)  # (2m,)
    distance = squared[:, None] + squared[None, :] - 2 * features @ features.T  # (2m, 2m)
    np.fill_diagonal(distance, np.inf)
    neighbours = np.argpartition(distance, k, axis=1)[:, :k]  # (2m, k)
    predicted = labels[neighbours].mean(axis=1) > 0.5  # (2m,), k odd, so never a tie
    return float((predicted == labels).mean())


split_rng = np.random.default_rng(1)
SPLITS = [split_rng.permutation(DATASETS) for _ in range(20)]


def c2st(first: str, second: str) -> str:
    accuracies = np.array(
        [knnAccuracy(TABLES[first][split[:half]], TABLES[second][split[half:]]) for split in SPLITS]
    )
    return f'{accuracies.mean():.3f} [{accuracies.min():.3f}, {accuracies.max():.3f}]'


pairs = [(pooling, 'today') for pooling in POOLINGS]
pairs += [
    ('independent', 'independent'),
    ('alike', 'independent'),
    ('share', 'independent'),
    ('alike', 'share'),
]
for first, second in pairs:
    print(f'{first:>12} vs {second:<12} {c2st(first, second)}')

       today vs today        0.497 [0.460, 0.523]


 independent vs today        0.543 [0.519, 0.569]


       alike vs today        0.518 [0.492, 0.552]


       share vs today        0.529 [0.492, 0.553]


      median vs today        0.603 [0.588, 0.630]


 independent vs independent  0.497 [0.470, 0.530]


       alike vs independent  0.494 [0.465, 0.530]


       share vs independent  0.501 [0.473, 0.547]


       alike vs share        0.497 [0.474, 0.516]


## Code cost

Each design built as a sketch on a scratch copy of `src/` (never on `src/` itself), formatted with ruff,
and run for 300 datasets: all values finite, per-unit values recorded, an overridden point kept as it is.
The line counts are from `git diff --no-index` against the working tree.
Tests are not counted: each sketch breaks 47 to 50 of the 135, through the new signatures of `apply` and the families,
the `Graph` fixtures, and `resolve` no longer returning points only.

| | A, independent | B-share | B-alike |
|---|---|---|---|
| lines added / removed, 7 files | 81 / 39 | 101 / 39 | 142 / 49 |
| new field | none | `alikeness`, Uniform(0, 1) | `alikeness`, Uniform(0, 1) |
| new class | none | `Pooled` | `Pooled` |
| `resolve` leaves for the ten fields | their distributions | a `Pooled`: prior, centre, alikeness | the same |
| a `Beta` override of one of the ten | works | works | needs scipy's `betaincinv`, so excluded |
| exact ties among a dataset's recorded values | none | by design, at chance ρ | none |

### A, independent

- `hyperprior.py`: a class constant `PER_UNIT` names the ten fields; `resolve` skips them, so `Dataset.hyperparameters` keeps their distributions;
  `drawEach` draws one unit's value, or returns an overridden point.
- `marginal.py`: `PARAMETERS`, the fields each family reads; a family takes `(n, parameters, rng)` instead of `resolved`.
- `dataset.py`: `Column.parameters`, the column's own draws; `Graph.noise_scale` and `Graph.init_scale`, `(n,)` each.
- `column.py`: `_column` draws a column's parameters; `apply` and `_codes` read them and drop `resolved`.
- `graph.py`: `connect` draws a noise and an init scale per node; `mechanism.py` reads them from the graph, `_mlp` takes `init_scale` and the width.

```python
# hyperprior.py
def resolve(self, rng):
    points = {}
    for name in type(self).model_fields:
        value = getattr(self, name)
        drawn = isinstance(value, Distribution) and name not in self.PER_UNIT
        points[name] = value.draw(rng) if drawn else value
    return Hyperprior(**points)

def drawEach(self, name, rng):
    value = getattr(self, name)
    return value.draw(rng) if isinstance(value, Distribution) else value

# column.py, _column
marginal = resolved.marginal_shares.choose(rng)
parameters = {name: resolved.drawEach(name, rng) for name in PARAMETERS[marginal]}
return Column(type=type, marginal=marginal, parameters=parameters)

# graph.py, connect
noise_scale = np.array([resolved.drawEach('noise_scale', rng) for _ in range(n)])  # (n,)
init_scale = np.array([resolved.drawEach('init_scale', rng) for _ in range(n)])  # (n,)
```

### B-share: A and 20 lines more

`resolve` draws every field as today, then wraps each of the ten still a distribution in a `Pooled`,
copied in after validation because a `Pooled` is no field's type. `drawEach` is A's: a `Pooled` is a `Distribution`.

```python
class Pooled(Distribution):
    def __init__(self, prior, centre, alikeness): ...

    def draw(self, rng):
        own = self.prior.draw(rng)
        return self.centre if rng.random() < self.alikeness else own

# Hyperprior.resolve, after the loop of today
resolved = Hyperprior(**points)
pooled = {}
for name in self.PER_UNIT:
    prior = getattr(self, name)
    if isinstance(prior, Distribution):
        pooled[name] = Pooled(prior, points[name], resolved.alikeness)
return resolved.model_copy(update=pooled)
```

### B-alike: B-share and 41 lines more

`cdf` and `quantile` on `Uniform` and `LogUniform`, the copula in `Pooled.draw` with `statistics.NormalDist`,
and the ten fields typed `float | Uniform | LogUniform`. About 12 of the 41 lines are ruff wrapping the longer type names.

```python
def draw(self, rng):
    z_centre = self.NORMAL.inv_cdf(self.prior.cdf(self.centre))
    noise = math.sqrt(1 - self.alikeness**2) * rng.standard_normal()
    return self.prior.quantile(self.NORMAL.cdf(self.alikeness * z_centre + noise))
```

## Verdict

**Visible, but small, and more inside the tables than between them.**

- **Parameter level.** A cuts how much datasets differ in their mean log noise scale from 1.34 to 0.58,
  and spreads a dataset's nodes by 1.28, close to D's own 1.33.
  The two-level designs sit midway: 0.89 and 0.90 between datasets, 1.01 and 0.94 within.
  `lognormal_sigma` barely moves under any design (0.51 to 0.42–0.45 between):
  a dataset has 1.05 lognormal-family columns on average, so there is little to pool.
- **Each unit's law is D** under every design.
  In the pipeline, one first-reader distance of 40 lies above its bar (`share`, `zero_share`, 0.064 against 0.057);
  its first readers share their centres with `today`'s, which sits at the bar too, so the two are one deviation of those 567 centres.
  Straight from `spread`, 100,000 draws put all six distances under 0.0043.
- **Table level.** A shrinks the between-table sd of mean |Spearman| by 12 %, of mean skewness by 19 %,
  of mean kurtosis by 27 % and of the mean largest share by 5 %;
  B-alike by 9, 15, 27 and 1 %, B-share by 9, 13, 24 and 4 %.
  What grows is the spread within a table: on average, the sd of skewness across its continuous columns
  from 1.33 to 1.63, of kurtosis from 16 to 21, of the largest share from 0.145 to 0.161.
- **The graph washes it out for Spearman, the family shares for the marginals.**
  Without any variation of the ten fields (`median`), the Spearman spread stays as it is (0.200 against 0.197):
  the graph sets it, so A's 56 % cut of the noise-scale spread arrives as 12 %.
  The ten fields do carry most of the skewness spread (0.65 without them against 1.75),
  but A cuts it by only 19 %, because a family seldom has more than one column in a table.
- **A classifier sees it, barely.** The k-NN C2ST tells A from today at 0.543 against a null of 0.497,
  1,000 tables a side; the two-level designs at 0.518 and 0.529, the `median` floor at 0.603.
  The per-summary KS distances show where: the within-table sds of skewness (0.14), kurtosis (0.15)
  and the largest share (0.13), against about 0.05 between independent halves of today.
  At their default alikeness, Uniform(0, 1), both two-level designs are indistinguishable from A (0.494 and 0.501)
  and from each other (0.497).
- **Code.** A adds 81 lines and removes 39 over 7 files, with no new field.
  B-share needs 20 lines more, a field and a class; B-alike 61 more than A, and it loses the `Beta` override without scipy.

**Recommendation**, for #16 to decide: A.
At their default, the two-level designs draw tables a classifier cannot tell from A's,
so their extra field buys nothing the proofs would see.
Until item 4's fit shows the registry's tables more alike within than A draws them, it is speculative generality.
If the fit does show that, B-share is the 20-line addition: it nests A at alikeness 0 and today at 1, and takes any distribution.
B-alike is dominated: B-share's tables at three times its extra code, and without `Beta`.

**Decision.** A, taken on #16 (2026-10-12): each column or node draws its own value of the ten fields,
and no alikeness field is added until item 4's fit asks for one; B-share is the design kept in reserve.